In [31]:
import os
import pandas as pd
import numpy as np
import os
from PIL import Image

import torch
from torch.utils.data import Dataset
from torchvision import transforms

In [32]:
os.chdir("C:/Users/moham/Downloads/Projects/Breast_Cancer_Prediction/Dataset/Breast_Cancer_Final")
os.listdir()

['test', 'test.csv', 'train', 'train.csv']

In [33]:
PROJECT_ROOT = r"C:\Users\...\Breast_Cancer_Final"

train_csv = os.path.join(PROJECT_ROOT, "train.csv")
test_csv  = os.path.join(PROJECT_ROOT, "test.csv")

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [34]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.ToTensor()
])

In [35]:
from PIL import Image
from torch.utils.data import Dataset
import torch

class BreastCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = self.dataframe.loc[idx, "filepath"]
        label = self.dataframe.loc[idx, "label"]

        image = Image.open(image_path).convert("L")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [36]:
train_dataset = BreastCancerDataset(
    dataframe=train_df,
    transform=train_transform
)

test_dataset = BreastCancerDataset(
    dataframe=test_df,
    transform=test_transform
)

In [37]:
print("Training images :", len(train_dataset))
print("Testing images  :", len(test_dataset))

Training images : 1164
Testing images  : 299


In [38]:
image, label = train_dataset[0]

print(image.shape)
print(label)

torch.Size([1, 512, 512])
tensor(0)


In [39]:
from PIL import Image

img = Image.open(train_df.loc[0, "filepath"])

print(img.mode)
print(img.size)

L
(512, 512)


In [40]:
from torch.utils.data import DataLoader

In [41]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [42]:
images, labels = next(iter(train_loader))

print("Images Shape :", images.shape)
print("Labels Shape :", labels.shape)

c:\Users\moham\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Images Shape : torch.Size([16, 1, 512, 512])
Labels Shape : torch.Size([16])


In [43]:
pin_memory=False

In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BreastCancerCNN(nn.Module):

    def __init__(self):
        super(BreastCancerCNN, self).__init__()

        # -------- Feature Extractor --------
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.conv4 = nn.Conv2d(
            in_channels=128,
            out_channels=256,
            kernel_size=3,
            stride=1,
            padding=1
        )

        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout
        self.dropout1 = nn.Dropout(0.5)
        self.dropout2 = nn.Dropout(0.3)

        # Classifier
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):

        # Block 1
        x = self.pool(F.relu(self.conv1(x)))

        # Output:
        # 32 x 256 x 256

        # Block 2
        x = self.pool(F.relu(self.conv2(x)))

        # Output:
        # 64 x 128 x 128

        # Block 3
        x = self.pool(F.relu(self.conv3(x)))

        # Output:
        # 128 x 64 x 64

        # Block 4
        x = F.relu(self.conv4(x))

        # Output:
        # 256 x 64 x 64

        # Global Average Pooling
        x = self.gap(x)

        # Output:
        # 256 x 1 x 1

        # Flatten
        x = torch.flatten(x, 1)

        # Output:
        # 256

        x = self.dropout1(x)

        x = F.relu(self.fc1(x))

        # Output:
        # 64

        x = self.dropout2(x)

        x = self.fc2(x)

        # Output:
        # 2

        return x

In [46]:
model = BreastCancerCNN()
print(model)

BreastCancerCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (gap): AdaptiveAvgPool2d(output_size=(1, 1))
  (dropout1): Dropout(p=0.5, inplace=False)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)


In [47]:
outputs = model(images)

print(outputs.shape)

torch.Size([16, 2])


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f}")

Epoch [1/10] | Loss: 0.6947
Epoch [2/10] | Loss: 0.6931
